# Notebook 06 — Database Pipeline & SQL Analytics

**Purpose:** Demonstrate the full data engineering lifecycle:
1. Initialize the database schema
2. Ingest datasets via the ETL pipeline
3. Run analytical SQL queries
4. Manage users and roles
5. Generate and query alerts

**Skills demonstrated:** Data Engineering, SQL analytics, user management, platform integration

> This notebook works standalone — it creates a local SQLite database. No prior notebooks required.
> To use PostgreSQL, set `DATABASE_URL` in `.env` before running.

In [2]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import text
import json

from src.config import Paths, db_settings
from src.db.connector import DatabaseManager
from src.db.models import User, UserRole, Alert, AlertSeverity, AlertStatus, ModelRun

print(f'Database URL: {db_settings.database_url}')

Database URL: sqlite:///./network_security.db


## 1. Initialize Database

In [3]:
# Create all tables and bootstrap admin user
mgr = DatabaseManager()
mgr.create_tables()
admin = mgr.bootstrap_admin()

print(f'Database initialized.')
print(f'Admin user: {admin.username} (role: {admin.role})')
print(f'\nHealth check: {mgr.health_check()}')

2026-03-29 16:26:08.646 | INFO     | src.db.connector:__init__:70 - DatabaseManager connected → sqlite:///./network_security.db
2026-03-29 16:26:08.648 | INFO     | src.db.connector:create_tables:85 - All tables created (or already exist).
2026-03-29 16:26:08.659 | INFO     | src.db.connector:bootstrap_admin:129 - Admin user 'admin' already exists.


Database initialized.
Admin user: admin (role: UserRole.admin)

Health check: {'status': 'ok', 'engine': 'sqlite:///./network_security.db'}


## 2. Create Users (Role-Based Access Control)

In [4]:
# Create sample users for each role
sample_users = [
    ('alice_analyst', 'alice@soc.example.com', 'SecurePass123!', UserRole.analyst),
    ('bob_ds', 'bob@datascience.example.com', 'SecurePass123!', UserRole.data_scientist),
    ('carol_viewer', 'carol@example.com', 'SecurePass123!', UserRole.viewer),
]

created = []
with mgr.get_session() as session:
    for username, email, password, role in sample_users:
        existing = session.query(User).filter_by(username=username).first()
        if not existing:
            user = mgr.create_user(username=username, email=email,
                                   password=password, role=role)
            created.append(user.username)
        else:
            created.append(f'{username} (already exists)')

print('Users created:', created)

# Show all users
users = mgr.list_users()
df_users = pd.DataFrame([
    {'id': u.id, 'username': u.username, 'role': u.role.value,
     'active': u.is_active, 'created': u.created_at}
    for u in users
])
print(f'\nTotal users in database: {len(df_users)}')
df_users

2026-03-29 16:26:08.854 | INFO     | src.db.connector:create_user:163 - User created: alice_analyst (UserRole.analyst)
2026-03-29 16:26:09.023 | INFO     | src.db.connector:create_user:163 - User created: bob_ds (UserRole.data_scientist)
2026-03-29 16:26:09.192 | INFO     | src.db.connector:create_user:163 - User created: carol_viewer (UserRole.viewer)


Users created: ['alice_analyst', 'bob_ds', 'carol_viewer']

Total users in database: 4


,id,username,role,active,created
0,1,admin,admin,True,2026-03-29 20:23:48
1,2,alice_analyst,analyst,True,2026-03-29 20:26:08
2,3,bob_ds,data_scientist,True,2026-03-29 20:26:09
3,4,carol_viewer,viewer,True,2026-03-29 20:26:09


## 3. ETL: Ingest UNSW-NB15 (Sample)

Full ingestion via `python -m src.db.ingest --all`. Here we demonstrate with a 5,000-row sample.

In [5]:
from src.db.models import NetworkEvent, DatasetRegistry
from datetime import datetime

SAMPLE_ROWS = 5000

if Paths.UNSW_TRAIN.exists():
    df_sample = pd.read_csv(Paths.UNSW_TRAIN, nrows=SAMPLE_ROWS, low_memory=False)
    df_sample.columns = df_sample.columns.str.strip().str.lower()
    print(f'Loaded {len(df_sample)} rows from UNSW-NB15 train')

    with mgr.get_session() as session:
        # Register dataset
        existing = session.query(DatasetRegistry).filter_by(name='unsw_nb15_sample').first()
        if not existing:
            registry = DatasetRegistry(
                name='unsw_nb15_sample',
                source_file=str(Paths.UNSW_TRAIN),
                row_count=SAMPLE_ROWS,
                column_count=len(df_sample.columns),
                description='UNSW-NB15 training sample (5,000 rows for demo)',
            )
            session.add(registry)
            session.flush()
            dataset_id = registry.id
        else:
            dataset_id = existing.id

        # Check if already ingested
        existing_count = session.query(NetworkEvent).filter_by(
            dataset_id=dataset_id).count()
        
        if existing_count == 0:
            records = []
            for _, row in df_sample.iterrows():
                records.append(NetworkEvent(
                    dataset_id=dataset_id, split='train',
                    proto=str(row.get('proto', ''))[:20],
                    service=str(row.get('service', ''))[:20],
                    state=str(row.get('state', ''))[:20],
                    dur=float(row['dur']) if pd.notna(row.get('dur')) else None,
                    sbytes=int(row['sbytes']) if pd.notna(row.get('sbytes')) else None,
                    dbytes=int(row['dbytes']) if pd.notna(row.get('dbytes')) else None,
                    sttl=int(row['sttl']) if pd.notna(row.get('sttl')) else None,
                    dttl=int(row['dttl']) if pd.notna(row.get('dttl')) else None,
                    spkts=int(row['spkts']) if pd.notna(row.get('spkts')) else None,
                    dpkts=int(row['dpkts']) if pd.notna(row.get('dpkts')) else None,
                    label=int(row['label']) if pd.notna(row.get('label')) else None,
                    attack_cat=str(row.get('attack_cat', ''))[:30],
                ))
            session.bulk_save_objects(records)
            session.commit()
            print(f'Inserted {len(records):,} network event records.')
        else:
            print(f'Already have {existing_count:,} records for this dataset.')
else:
    print('UNSW-NB15 data not found — skipping ingestion demo.')
    print(f'Expected at: {Paths.UNSW_TRAIN}')

Loaded 5000 rows from UNSW-NB15 train
Inserted 5,000 network event records.


## 4. SQL Analytics Queries

Demonstrating the analyst workflow: exploring the database with SQL.

In [6]:
def query(sql):
    with mgr.engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

# ── Query 1: Attack distribution ──────────────────────────────────────────────
q1 = query("""
    SELECT 
        attack_cat,
        COUNT(*) AS total,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM network_events
    WHERE label = 1
    GROUP BY attack_cat
    ORDER BY total DESC
""")
print('Attack category distribution:')
print(q1.to_string(index=False))

Attack category distribution:
Empty DataFrame
Columns: [attack_cat, total, pct]
Index: []


In [7]:
# ── Query 2: Protocol breakdown ───────────────────────────────────────────────
q2 = query("""
    SELECT 
        proto,
        COUNT(*) AS total_flows,
        SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END) AS attacks,
        ROUND(100.0 * SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS attack_pct
    FROM network_events
    GROUP BY proto
    ORDER BY total_flows DESC
    LIMIT 10
""")
print('Attack rate by protocol:')
print(q2.to_string(index=False))

Attack rate by protocol:
proto  total_flows  attacks  attack_pct
  tcp         3934        0         0.0
  udp         1040        0         0.0
  arp           14        0         0.0
 ospf           10        0         0.0
 icmp            2        0         0.0


In [8]:
# ── Query 3: Traffic volume stats ─────────────────────────────────────────────
q3 = query("""
    SELECT
        CASE WHEN label = 1 THEN 'Attack' ELSE 'Normal' END AS traffic_type,
        COUNT(*) AS n_flows,
        ROUND(AVG(sbytes), 0) AS avg_src_bytes,
        ROUND(AVG(dbytes), 0) AS avg_dst_bytes,
        ROUND(AVG(dur), 4) AS avg_duration,
        ROUND(AVG(spkts), 1) AS avg_src_pkts
    FROM network_events
    GROUP BY label
""")
print('Traffic statistics by class:')
print(q3.to_string(index=False))
print('\nInsight: Attack traffic tends to differ in packet counts and byte volumes.')

Traffic statistics by class:
traffic_type  n_flows  avg_src_bytes  avg_dst_bytes  avg_duration  avg_src_pkts
      Normal     5000         2866.0        21020.0        1.8796          26.3

Insight: Attack traffic tends to differ in packet counts and byte volumes.


## 5. Generate Threat Alerts from Query Results

In [9]:
from src.db.models import Alert

# Simulate alerts generated by a detection model
alert_templates = [
    dict(title='High-volume DoS attack detected',
         severity=AlertSeverity.critical, attack_type='DoS',
         confidence=0.94, source_dataset='unsw_nb15',
         description='Unusual packet rate detected from multiple source IPs. RF detector confidence: 94%.'),
    dict(title='Exploit attempt against HTTP service',
         severity=AlertSeverity.high, attack_type='Exploits',
         confidence=0.88, source_dataset='unsw_nb15',
         description='Abnormal payload patterns in HTTP flows match known exploit signatures.'),
    dict(title='Fuzzing activity detected on port 443',
         severity=AlertSeverity.medium, attack_type='Fuzzers',
         confidence=0.71, source_dataset='unsw_nb15',
         description='Sequential port probe pattern consistent with fuzzer activity.'),
    dict(title='Reconnaissance scan from external IP',
         severity=AlertSeverity.low, attack_type='Reconnaissance',
         confidence=0.65, source_dataset='unsw_nb15',
         description='Low-and-slow scan pattern detected over 15-minute window.'),
]

with mgr.get_session() as session:
    existing_alerts = session.query(Alert).count()
    if existing_alerts == 0:
        for template in alert_templates:
            alert = Alert(**template, status=AlertStatus.open)
            session.add(alert)
        session.commit()
        print(f'Created {len(alert_templates)} sample alerts.')
    else:
        print(f'Already have {existing_alerts} alerts in database.')

# Query alerts
df_alerts = query("""
    SELECT title, severity, status, attack_type, confidence, created_at
    FROM alerts
    ORDER BY 
        CASE severity 
            WHEN 'critical' THEN 1 WHEN 'high' THEN 2 
            WHEN 'medium' THEN 3 ELSE 4 
        END
""")
print('\nOpen alerts (priority order):')
df_alerts

Created 4 sample alerts.

Open alerts (priority order):


,title,severity,status,attack_type,confidence,created_at
0,High-volume DoS attack detected,critical,open,DoS,0.94,2026-03-29 20:26:09
1,Exploit attempt against HTTP service,high,open,Exploits,0.88,2026-03-29 20:26:09
2,Fuzzing activity detected on port 443,medium,open,Fuzzers,0.71,2026-03-29 20:26:09
3,Reconnaissance scan from external IP,low,open,Reconnaissance,0.65,2026-03-29 20:26:09


## 6. Alert Acknowledgment Workflow

In [10]:
from datetime import datetime

# Simulate an analyst acknowledging the critical alert
with mgr.get_session() as session:
    alice = session.query(User).filter_by(username='alice_analyst').first()
    critical_alert = session.query(Alert).filter_by(
        severity=AlertSeverity.critical).first()

    if critical_alert and alice:
        critical_alert.status = AlertStatus.acknowledged
        critical_alert.acknowledged_at = datetime.utcnow()
        critical_alert.acknowledged_by = alice.id
        session.commit()
        print(f'Alert acknowledged by: {alice.username}')
        print(f'Alert: {critical_alert.title}')
        print(f'Status: {critical_alert.status}')
    else:
        print('Create users (cell 2) and alerts (cell 5) first.')

# Show alert status summary
df_status = query("""
    SELECT severity, status, COUNT(*) as count
    FROM alerts
    GROUP BY severity, status
    ORDER BY CASE severity WHEN 'critical' THEN 1 WHEN 'high' THEN 2 
             WHEN 'medium' THEN 3 ELSE 4 END
""")
print('\nAlert status summary:')
print(df_status.to_string(index=False))

Alert acknowledged by: alice_analyst
Alert: High-volume DoS attack detected
Status: AlertStatus.acknowledged

Alert status summary:
severity       status  count
critical acknowledged      1
    high         open      1
  medium         open      1
     low         open      1


## 7. Database Summary

In [11]:
tables = ['users', 'dataset_registry', 'network_events', 
          'system_call_events', 'model_runs', 'predictions', 'alerts']

print('DATABASE SUMMARY')
print('='*40)
for table in tables:
    try:
        count = query(f'SELECT COUNT(*) as n FROM {table}').iloc[0, 0]
        print(f'  {table:<30} {count:>8,} rows')
    except Exception as e:
        print(f'  {table:<30} ERROR: {e}')

print('='*40)
print(f'\nDatabase file: {db_settings.database_url}')
print('\nNext steps:')
print('  streamlit run dashboard/app.py   → view in dashboard')
print('  uvicorn src.api.app:app --reload  → query via REST API')
print('  python -m src.db.ingest --all    → ingest full datasets')

DATABASE SUMMARY
  users                                 4 rows
  dataset_registry                      1 rows
  network_events                    5,000 rows
  system_call_events                    0 rows
  model_runs                           12 rows
  predictions                           0 rows
  alerts                                4 rows

Database file: sqlite:///./network_security.db

Next steps:
  streamlit run dashboard/app.py   → view in dashboard
  uvicorn src.api.app:app --reload  → query via REST API
  python -m src.db.ingest --all    → ingest full datasets
